In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# import os
# from PIL import Image

# base_path = "/content/drive/MyDrive/Chatbot/Routing_model_dataset"

# bad_files = 0

# for root, dirs, files in os.walk(base_path):
#     for file in files:
#         path = os.path.join(root, file)
#         try:
#             with Image.open(path) as img:
#                 img.verify()  # check if valid
#         except:
#             os.remove(path)
#             bad_files += 1

# print("Removed bad files:", bad_files)

In [3]:
# import os
# from PIL import Image

# base_path = "/content/drive/MyDrive/Chatbot/Routing_model_dataset"

# valid_ext = (".jpg", ".jpeg", ".png")
# bad_files = 0

# for root, dirs, files in os.walk(base_path):
#     for file in files:
#         path = os.path.join(root, file)

#         # 1. Remove wrong extensions immediately
#         if not file.lower().endswith(valid_ext):
#             try:
#                 os.remove(path)
#                 bad_files += 1
#             except:
#                 pass
#             continue

#         # 2. Validate image properly (full decode)
#         try:
#             with Image.open(path) as img:
#                 img = img.convert("RGB")  # forces full read
#         except:
#             try:
#                 os.remove(path)
#                 bad_files += 1
#             except:
#                 pass

# print("Removed bad files:", bad_files)

In [4]:
# import os
# import tensorflow as tf

# base_path = "/content/drive/MyDrive/Chatbot/Routing_model_dataset"

# bad_files = 0

# for root, dirs, files in os.walk(base_path):
#     for file in files:
#         path = os.path.join(root, file)

#         try:
#             img = tf.io.read_file(path)
#             img = tf.image.decode_image(img)  # same as training uses
#         except:
#             try:
#                 os.remove(path)
#                 bad_files += 1
#             except:
#                 pass

# print("Removed bad files:", bad_files)

In [5]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print("Using GPU:", gpus)
else:
    print("No GPU found")

Using GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [6]:
import tensorflow as tf
from tensorflow.keras import layers, models

data_dir = "/content/drive/MyDrive/Chatbot/Routing_model_dataset"

IMG_SIZE = 224
BATCH_SIZE = 32

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)

Found 15355 files belonging to 6 classes.
Using 12284 files for training.
Found 15355 files belonging to 6 classes.
Using 3071 files for validation.


In [7]:
import os

save_dir = "/content/drive/MyDrive/Chatbot/Routing_Models"
os.makedirs(save_dir, exist_ok=True)

In [8]:
def build_model():
    base = tf.keras.applications.EfficientNetB0(
        include_top=False,
        weights="imagenet",
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    base.trainable = False

    x = layers.GlobalAveragePooling2D()(base.output)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.4)(x)
    output = layers.Dense(len(class_names), activation="softmax")(x)

    model = tf.keras.Model(base.input, output)
    return model

model = build_model()

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


In [9]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [10]:
checkpoint = tf.keras.callbacks.ModelCheckpoint(
    filepath=save_dir + "/routing_model_epoch_{epoch:02d}.h5",
    save_weights_only=False,
    save_freq='epoch',
    verbose=1
)

In [11]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[checkpoint]
)

Epoch 1/10
384/384 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step - accuracy: 0.8725 - loss: 0.3903
Epoch 1: saving model to /content/drive/MyDrive/Chatbot/Routing_Models/routing_model_epoch_01.h5



Epoch 1: finished saving model to /content/drive/MyDrive/Chatbot/Routing_Models/routing_model_epoch_01.h5
384/384 ━━━━━━━━━━━━━━━━━━━━ 971s 555ms/step - accuracy: 0.9291 - loss: 0.2208 - val_accuracy: 0.9766 - val_loss: 0.0776
Epoch 2/10
382/384 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.9772 - loss: 0.0680
Epoch 2: saving model to /content/drive/MyDrive/Chatbot/Routing_Models/routing_model_epoch_02.h5



Epoch 2: finished saving model to /content/drive/MyDrive/Chatbot/Routing_Models/routing_model_epoch_02.h5
384/384 ━━━━━━━━━━━━━━━━━━━━ 10s 26ms/step - accuracy: 0.9759 - loss: 0.0739 - val_accuracy: 0.9775 - val_loss: 0.0756
Epoch 3/10
382/384 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.9819 - loss: 0.0550
Epoch 3: saving model to /content/drive/MyDrive/Chatbot/Routing_Models/routing_model_epoch_03.h5



Epoch 3: finished saving model to /content/drive/MyDrive/Chatbot/Routing_Models/routing_model_epoch_03.h5
384/384 ━━━━━━━━━━━━━━━━━━━━ 10s 26ms/step - accuracy: 0.9803 - loss: 0.0572 - val_accuracy: 0.9814 - val_loss: 0.0675
Epoch 4/10
382/384 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.9868 - loss: 0.0419
Epoch 4: saving model to /content/drive/MyDrive/Chatbot/Routing_Models/routing_model_epoch_04.h5



Epoch 4: finished saving model to /content/drive/MyDrive/Chatbot/Routing_Models/routing_model_epoch_04.h5
384/384 ━━━━━━━━━━━━━━━━━━━━ 10s 26ms/step - accuracy: 0.9883 - loss: 0.0378 - val_accuracy: 0.9814 - val_loss: 0.0738
Epoch 5/10
382/384 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.9877 - loss: 0.0347
Epoch 5: saving model to /content/drive/MyDrive/Chatbot/Routing_Models/routing_model_epoch_05.h5


...


Epoch 6: finished saving model to /content/drive/MyDrive/Chatbot/Routing_Models/routing_model_epoch_06.h5
384/384 ━━━━━━━━━━━━━━━━━━━━ 10s 26ms/step - accuracy: 0.9897 - loss: 0.0298 - val_accuracy: 0.9831 - val_loss: 0.0761
Epoch 7/10
382/384 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.9898 - loss: 0.0298
Epoch 7: saving model to /content/drive/MyDrive/Chatbot/Routing_Models/routing_model_epoch_07.h5



Epoch 7: finished saving model to /content/drive/MyDrive/Chatbot/Routing_Models/routing_model_epoch_07.h5
384/384 ━━━━━━━━━━━━━━━━━━━━ 10s 26ms/step - accuracy: 0.9894 - loss: 0.0306 - val_accuracy: 0.9821 - val_loss: 0.0802
Epoch 8/10
382/384 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.9894 - loss: 0.0317
Epoch 8: saving model to /content/drive/MyDrive/Chatbot/Routing_Models/routing_model_epoch_08.h5



Epoch 8: finished saving model to /content/drive/MyDrive/Chatbot/Routing_Models/routing_model_epoch_08.h5
384/384 ━━━━━━━━━━━━━━━━━━━━ 10s 26ms/step - accuracy: 0.9893 - loss: 0.0302 - val_accuracy: 0.9837 - val_loss: 0.0773
Epoch 9/10
382/384 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.9927 - loss: 0.0245
Epoch 9: saving model to /content/drive/MyDrive/Chatbot/Routing_Models/routing_model_epoch_09.h5



Epoch 9: finished saving model to /content/drive/MyDrive/Chatbot/Routing_Models/routing_model_epoch_09.h5
384/384 ━━━━━━━━━━━━━━━━━━━━ 10s 26ms/step - accuracy: 0.9925 - loss: 0.0239 - val_accuracy: 0.9840 - val_loss: 0.0851
Epoch 10/10
382/384 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.9941 - loss: 0.0188
Epoch 10: saving model to /content/drive/MyDrive/Chatbot/Routing_Models/routing_model_epoch_10.h5



Epoch 10: finished saving model to /content/drive/MyDrive/Chatbot/Routing_Models/routing_model_epoch_10.h5
384/384 ━━━━━━━━━━━━━━━━━━━━ 10s 26ms/step - accuracy: 0.9929 - loss: 0.0221 - val_accuracy: 0.9814 - val_loss: 0.0950


In [12]:
model.save(save_dir + "/routing_model_final.h5")

In [13]:
# import numpy as np

# def ensemble_predict(img_array):
#     p1 = model1.predict(img_array)
#     p2 = model2.predict(img_array)

#     final_pred = (p1 + p2) / 2
#     return np.argmax(final_pred)

In [14]:
# model1.save("/content/drive/MyDrive/Chatbot/routing_model_effnet.h5")
# model2.save("/content/drive/MyDrive/Chatbot/routing_model_resnet.h5")

In [15]:
# from tensorflow.keras.preprocessing import image

# img = image.load_img("test.jpg", target_size=(224,224))
# img_array = image.img_to_array(img) / 255.0
# img_array = np.expand_dims(img_array, axis=0)

# pred_class = ensemble_predict(img_array)
# print(class_names[pred_class])

In [16]:
# import tensorflow as tf
# from tensorflow.keras import layers

# input_layer = tf.keras.Input(shape=(224,224,3))

# p1 = model1(input_layer)
# p2 = model2(input_layer)

# avg = layers.Average()([p1, p2])

# ensemble_model = tf.keras.Model(inputs=input_layer, outputs=avg)

# ensemble_model.save("/content/drive/MyDrive/Chatbot/routing_ensemble_model.h5")